In [ ]:
from pathlib import Path
import time

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

BASE_DIR = Path("/home/eleroy/proj-dirs/fast-o3-iv/data/external/NCAR/CESM2/ssp585")
VARIABLE = "Z3"

URL_TEMPLATE = (
    "https://osdf-director.osg-htc.org/ncar/gdex/d651067/"
    "b.e21.BSSP585smbb.f09_g17.CESM2-LE/atm/proc/tseries/month_1/{var}/"
    "b.e21.BSSP585smbb.f09_g17.CESM2-LE.{member}.cam.h0.{var}.201501-206412.nc"
)


def make_session() -> requests.Session:
    s = requests.Session()
    retry = Retry(
        total=3,
        connect=3,
        read=0,
        status=3,
        other=0,
        backoff_factor=1.0,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
    )
    s.mount("https://", HTTPAdapter(max_retries=retry))
    s.headers.update(
        {
            "User-Agent": "fast-o3-iv-downloader/1.0",
            "Accept-Encoding": "identity",
            "Connection": "close",
        }
    )
    return s


def download_with_resume(url: str, out_path: Path, attempts: int = 20) -> None:
    tmp_path = out_path.with_suffix(out_path.suffix + ".part")

    for i in range(1, attempts + 1):
        start = tmp_path.stat().st_size if tmp_path.exists() else 0
        headers = {"Range": f"bytes={start}-"} if start > 0 else {}

        try:
            with make_session() as session:
                with session.get(
                    url,
                    stream=True,
                    timeout=(30, 600),
                    headers=headers,
                    allow_redirects=True,
                ) as r:
                    if start > 0:
                        if r.status_code == 200:
                            # Server ignored Range; restart from scratch
                            tmp_path.unlink(missing_ok=True)
                            start = 0
                        elif r.status_code != 206:
                            raise RuntimeError(
                                f"Expected 206 for resume, got {r.status_code}"
                            )

                    r.raise_for_status()

                    mode = "ab" if start > 0 else "wb"
                    with open(tmp_path, mode) as f:
                        for chunk in r.iter_content(chunk_size=1024 * 1024):
                            if chunk:
                                f.write(chunk)

            tmp_path.replace(out_path)
            return

        except Exception as e:
            partial_size = tmp_path.stat().st_size if tmp_path.exists() else 0
            print(f"Attempt {i}/{attempts} failed for {url}")
            print(f"{type(e).__name__}: {e}")
            print(f"partial size = {partial_size:,} bytes")

            if i == attempts:
                raise

            time.sleep(min(90, 2 ** i))


for n in range(1, 16):
    member = f"{n:03d}"
    url = URL_TEMPLATE.format(var=VARIABLE, member=member)
    fname = Path(url).name

    out_dir = BASE_DIR / member / VARIABLE
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / fname

    if out_path.exists() and out_path.stat().st_size > 0:
        print(f"Skipping existing: {out_path}")
        continue

    print(f"Downloading {url} -> {out_path}")
    download_with_resume(url, out_path, attempts=20)
    print("done")

    time.sleep(2)